# Evaluator-optimizer workflow with Pydantic AI

The evaluator-optimizer pattern uses a generator LLM and an evaluator LLM in a loop. The generator creates content, the evaluator checks it, and if it doesn't pass, feedback is sent back to the generator.

```mermaid
flowchart LR
    In([In]) --> Gen["Generator (LLM)"]
    Gen -- "Solution" --> Eval["Evaluator (LLM)"]
    Eval -- "Accepted" --> Out([Out])
    Eval -- "Rejected + Feedback" --> Gen
```

**Examples:**
- Content generation that must match certain guidelines (style, tone, language)
- Improving search results iteratively

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

## Vanilla workflow

We have three agents:
- **Generator**: Creates an initial article
- **Fixer**: Improves the article based on feedback
- **Evaluator**: Checks if the article meets criteria (British English, appropriate for young audience, no em dashes)

In [ ]:
class Evaluation(BaseModel):
    explanation: str = Field(
        description="Explain why the text matches or not the evaluation criteria"
    )
    feedback: str = Field(
        description="Provide feedback to the writer to improve the text"
    )
    is_correct: bool = Field(
        description="Whether the text matches the evaluation criteria"
    )


generator = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert writer. Provided with a topic, "
        "you will generate an engaging article with less than 500 words."
    ),
)

fixer = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert writer. Provided with a text and feedback, "
        "you will improve the text."
    ),
)

evaluator = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert evaluator. Provided with a text, you will evaluate if it's written in "
        "British English and if it's appropriate for a young audience. The text must always use "
        "British spelling and grammar. Make sure the text doesn't include any em dashes."
    ),
    output_type=Evaluation,
)


def run_workflow(topic: str) -> str:
    text = generator.run_sync(f"Generate an article about '{topic}'").output

    for _ in range(3):
        evaluation = evaluator.run_sync(f"Evaluate the following text: {text}").output

        if evaluation.is_correct:
            return text

        text = fixer.run_sync(
            f"Fix the text: {text} with the following feedback: {evaluation.feedback}"
        ).output

    return text


output = run_workflow("Substance abuse of athletes")
print(output)

## Exercise

Transform the prompt chain workflow that generates recipes into an evaluator-optimizer workflow. It should make sure that the recipe is accurate, easy to follow, and that it has few ingredients.